# ChequeSense: Dataset Visualization & Preprocessing Showcase

This notebook provides interactive exploration, visualization, and preprocessing verification across all three datasets in **ChequeSense**:
1. **Synthetic Cheques** (Field localization with bounding boxes & bank distributions)
2. **Handwritten Lines & Cheque VQA** (Cursive handwriting transcription & amount discrepancy inspection)
3. **IDRBT 300 DPI Cheques** (Real-world Indian banking scans & gold-standard evaluation entities)
4. **Modular Image Preprocessor** (Deskewing, CLAHE contrast enhancement, Otsu binarization, and aspect-ratio padding)

In [ ]:
import io
import json
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import pandas as pd
from PIL import Image

from src.data.dataset_loader import DatasetLoader
from src.data.image_preprocessor import ImagePreprocessor, PreprocessingConfig

# Set plotting aesthetics
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10
print("ChequeSense modules imported successfully.")

## 1. Load Dataset Manifests and Summary
We inspect the machine-readable manifest generated by `ManifestBuilder`.

In [ ]:
manifest_df = pd.read_csv('data/manifests/dataset_manifest.csv')
print(f"Total dataset records in manifest: {len(manifest_df)}")
print("\nDistribution across splits and datasets:")
print(pd.crosstab(manifest_df['dataset_name'], manifest_df['split'], margins=True))

## 2. Visualize Synthetic Cheques with Bounding Box Annotations
Here we visualize a synthetic cheque with its 6 annotated ground-truth regions:
- `date` (Cyan)
- `amount` (Green)
- `ifsc` (Orange)
- `acno` (Magenta)
- `sign` (Red)
- `name` (Yellow)

In [ ]:
loader = DatasetLoader()
syn_samples = list(loader.load_synthetic(split='train', lazy=True))
sample = syn_samples[0]

img = sample.get_image()
fig, ax = plt.subplots(figsize=(14, 7))
ax.imshow(img)

color_palette = {
    'date': 'cyan',
    'amount': '#00FF00',
    'ifsc': 'orange',
    'acno': 'magenta',
    'sign': 'red',
    'name': 'yellow'
}

for field_name, coords in sample.annotations.items():
    xmin, ymin = coords['xmin'], coords['ymin']
    xmax, ymax = coords['xmax'], coords['ymax']
    w = xmax - xmin
    h = ymax - ymin
    col = color_palette.get(field_name, 'blue')
    rect = patches.Rectangle((xmin, ymin), w, h, linewidth=2, edgecolor=col, facecolor='none')
    ax.add_patch(rect)
    ax.text(xmin, ymin - 8, field_name.upper(), color=col, fontsize=10, weight='bold', bbox=dict(facecolor='black', alpha=0.6, pad=1))

ax.set_title(f"Synthetic Cheque Sample: {sample.sample_id} ({sample.metadata.get('bank', '').upper()}) - Resolution: {sample.width}x{sample.height}")
ax.axis('off')
plt.tight_layout()
plt.show()

## 3. Visualize Handwritten Text Lines (HTR Task)
The handwritten dataset contains line-level handwriting samples from the IAM benchmark for training cursive text recognition engines.

In [ ]:
hw_samples = list(loader.load_handwritten_and_cheques(split='train', category='handwritten_ocr', lazy=True))

fig, axes = plt.subplots(4, 1, figsize=(14, 8))
for i, ax in enumerate(axes):
    s = hw_samples[i]
    im = s.get_image()
    ax.imshow(im, cmap='gray')
    gt_text = s.annotations['ground_truth'].get('text', '')
    ax.set_title(f"Sample {s.sample_id} | Ground Truth: \"{gt_text}\"", fontsize=11, loc='left')
    ax.axis('off')

plt.tight_layout()
plt.show()

## 4. Visualize Cheque VQA & Amount Discrepancies
Here we inspect downscaled cheque VQA samples and observe deliberate amount discrepancies between legal amount in words and courtesy amount in figures.

In [ ]:
vqa_samples = list(loader.load_handwritten_and_cheques(split='train', category='cheque_vqa', lazy=True))
sample_vqa = vqa_samples[0]

fig, ax = plt.subplots(figsize=(10, 5))
ax.imshow(sample_vqa.get_image(), cmap='gray')
ax.set_title(f"Cheque VQA Sample: {sample_vqa.sample_id}")
ax.axis('off')
plt.show()

print("VQA Instruction Prompt:")
print(sample_vqa.annotations['query'])
print("\nExtracted Ground-Truth Fields:")
print(json.dumps(sample_vqa.annotations['ground_truth'], indent=2))

## 5. Visualize IDRBT Real Cheques & Linked Entity Labels
High-resolution 300 DPI real Indian banking cheque from IDRBT.

In [ ]:
idrbt_samples = list(loader.load_idrbt(lazy=True))
sample_idrbt = idrbt_samples[0]

fig, ax = plt.subplots(figsize=(14, 7))
ax.imshow(sample_idrbt.get_image())
ax.set_title(f"Real IDRBT Cheque Scan: {sample_idrbt.metadata['raw_filename']} ({sample_idrbt.width}x{sample_idrbt.height}, 300 DPI)")
ax.axis('off')
plt.tight_layout()
plt.show()

## 6. Image Preprocessing Pipeline Showcase
Demonstrating the modular image preprocessor operations:
1. **Original RGB**
2. **Bilateral Denoising**
3. **CLAHE Contrast Enhancement**
4. **Otsu Binarization**
5. **Aspect-Ratio Resizing with Margin Padding**

In [ ]:
test_img = sample.get_image()

# 1. Denoised
cfg_denoise = PreprocessingConfig(color_mode='grayscale', denoise=True, denoise_method='bilateral')
res_denoise = ImagePreprocessor(cfg_denoise).process(test_img)

# 2. CLAHE
cfg_clahe = PreprocessingConfig(color_mode='grayscale', enhance_contrast=True, contrast_method='clahe', clahe_clip_limit=3.0)
res_clahe = ImagePreprocessor(cfg_clahe).process(test_img)

# 3. Otsu Threshold
cfg_otsu = PreprocessingConfig(color_mode='grayscale', threshold_mode='otsu')
res_otsu = ImagePreprocessor(cfg_otsu).process(test_img)

# 4. Resized with Padding to square 800x800
cfg_resize = PreprocessingConfig(target_size=(800, 800), maintain_aspect_ratio=True)
res_resize = ImagePreprocessor(cfg_resize).process(test_img)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].imshow(res_denoise.image, cmap='gray')
axes[0, 0].set_title("Grayscale + Bilateral Denoising")
axes[0, 0].axis('off')

axes[0, 1].imshow(res_clahe.image, cmap='gray')
axes[0, 1].set_title("Grayscale + CLAHE Contrast Enhancement")
axes[0, 1].axis('off')

axes[1, 0].imshow(res_otsu.image, cmap='gray')
axes[1, 0].set_title("Otsu Binarization (Thresholding)")
axes[1, 0].axis('off')

axes[1, 1].imshow(res_resize.image)
axes[1, 1].set_title(f"Aspect-Ratio Preserved Resizing (Padded to 800x800)")
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()